In [43]:
#INSTALL
%pip install pandas
%pip install seaborn
%pip install spacy
%pip install fr_core_news_sm
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
#IMPORT
import pandas as pd
import seaborn as sns
import spacy
from spacy.lang.fr.stop_words import STOP_WORDS as fr_stop
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB, BernoulliNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [56]:
train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

stop_words_cuisine = [ # Générés par Gemini
    # --- Unités et Mesures ---
    "g", "gr", "gramme", "grammes", "kg", "kilogramme",
    "l", "litres", "litre", "ml", "cl", "dl",
    "cuillère", "cuillères", "c.à.s", "c.à.c", "cas", "cac",
    "pincée", "pincées", "tasse", "tasses", "verre", "verres",
    "poignée", "tranche", "tranches", "morceau", "morceaux",
    "bouteille", "sachet", "boîte", "pot", "paquet", "brin", 
    "feuille", "feuilles", "gousse", "gousses", "°","c", "mn",
    
    # --- Verbes d'action (Instructions fréquentes) ---
    "ajouter", "ajoutez", "mélanger", "mélangez", "remuer",
    "verser", "versez", "incorporer", "couper", "coupez",
    "hacher", "éplucher", "cuire", "laisser", "mettre", "mettez",
    "servir", "disposer", "saupoudrer", "garnir", "préparer",
    "chauffer", "préchauffer", "bouillir", "mijoter", "réserver",
    "laver", "égoutter", "battre", "fouetter", "étaler", "faire",
    
    # --- Adjectifs / États ---
    "chaud", "froide", "tiède", "bouillant", 
    "fondu", "haché", "râpé", "entier", "entière",
    "fin", "fine", "épais", "gros", "petit",
    "frais", "fraîche", "sec", "sèche",
    
    # --- Ingrédients "neutres" (souvent exclus de l'analyse thématique) ---
    # À retirer de cette liste si vous voulez garder tous les ingrédients
    "sel", "poivre", "eau", "huile", "beurre", "sucre", "farine",
    
    # --- Mots de liaison temporels ---
    "puis", "ensuite", "enfin", "pendant", "environ", "minutes", "min", 
    "heure", "heures", "seconde", "secondes", "avant", "après",

    # --- Autres ---
    " ", "1/2","bien"
]

stops = list(set(list(fr_stop)+stop_words_cuisine))

X_train = train_df["titre"] + " " + train_df["ingredients"] + " " + train_df["recette"]
X_test = test_df["titre"] + " " + test_df["ingredients"] + " " + test_df["recette"]
Y_train = train_df["type"]
Y_test = test_df["type"]

In [46]:
# Function to print results
def get_results(actual,predictions) :
    print(f"Resultat (Micro-F1) : {f1_score(actual, predictions, average='micro'):.2f}")
    print(f"Résultat (Macro-F1) : {f1_score(actual, predictions, average='macro'):.2f}")
    print(f"Résultat (Accuracy) : {accuracy_score(actual, predictions):.2f}")

    print(f"Classes : {sorted(actual.unique())}")
    print(f"Résultat (F1 par classe) : {f1_score(actual, predictions, average=None)}")
    print(f"Matrice de confusion : {confusion_matrix(actual, predictions)}")

In [57]:
# Méthode A : TF-IDF + Naive Bayes
model = Pipeline([  # sequentially apply a list of transformers
    ('tfidf', TfidfVectorizer(stop_words=stops, lowercase=True)),   # TF-IDF to preprocess the data
    ('cnb', ComplementNB()) # Naive Bayes to predict the class
])

model.fit(X_train, Y_train)

predictions = model.predict(X_test)

get_results(Y_test, predictions)

c:\Main_Doc\M1\S2\TAL\projet\tal_project\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['neuf', 'qu', 'quelqu'] not in stop_words.
  warnings.warn(


Resultat (Micro-F1) : 0.80
Résultat (Macro-F1) : 0.74
Résultat (Accuracy) : 0.80
Classes : ['Dessert', 'Entrée', 'Plat principal']
Résultat (F1 par classe) : [0.93995381 0.42920354 0.83813443]
Matrice de confusion : [[407   0   0]
 [ 37  97 203]
 [ 15  18 611]]


Nous remarquons que les desserts sont plutôt bien classifiés mais que notre modèle a plus de difficultés à différencier les entrées et les plats principaux.

In [61]:
# Méthode B-1 :  TF-IDF + 2-grammes + SVM
model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=stops, lowercase=True, ngram_range=(1,2))),
    ('svm', SVC(kernel='linear'))
])

model.fit(X_train, Y_train)
predictions = model.predict(X_test)
get_results(Y_test, predictions)

c:\Main_Doc\M1\S2\TAL\projet\tal_project\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['neuf', 'qu', 'quelqu'] not in stop_words.
  warnings.warn(


Resultat (Micro-F1) : 0.88
Résultat (Macro-F1) : 0.87
Résultat (Accuracy) : 0.88
Classes : ['Dessert', 'Entrée', 'Plat principal']
Résultat (F1 par classe) : [0.98663426 0.73584906 0.87471526]
Matrice de confusion : [[406   1   0]
 [  6 234  97]
 [  4  64 576]]


In [62]:
# Méthode B-2 :  TF-IDF + 2-grammes + logistic regression
model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=stops, lowercase=True, ngram_range=(1,2))),
    ('logreg', LogisticRegression(max_iter=1000))
])
model.fit(X_train, Y_train)
predictions = model.predict(X_test)
get_results(Y_test, predictions)

c:\Main_Doc\M1\S2\TAL\projet\tal_project\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['neuf', 'qu', 'quelqu'] not in stop_words.
  warnings.warn(


Resultat (Micro-F1) : 0.86
Résultat (Macro-F1) : 0.84
Résultat (Accuracy) : 0.86
Classes : ['Dessert', 'Entrée', 'Plat principal']
Résultat (F1 par classe) : [0.98186215 0.68341709 0.86538462]
Matrice de confusion : [[406   1   0]
 [ 10 204 123]
 [  4  55 585]]


In [63]:
# Méthode C : Modèle enrichi
X_train_3 = pd.DataFrame({
    "recette_complete": X_train,
    "nb_ingredients": train_df["ingredients"].apply(lambda x: len(x.split(" - "))),
    "length_recette": train_df["recette"].apply(lambda x: len(x))
})
X_test_3 = pd.DataFrame({
    "recette_complete": X_test,
    "nb_ingredients": test_df["ingredients"].apply(lambda x: len(x.split(" - "))),
    "length_recette": test_df["recette"].apply(lambda x: len(x))
})
Y_train= train_df["type"]
Y_test = test_df["type"]

model = Pipeline([
    ('features', ColumnTransformer([
        ('tfidf', TfidfVectorizer(stop_words=stops, lowercase=True), 'recette_complete'),
        ('num', MinMaxScaler(), ['nb_ingredients', 'length_recette'])
    ])),
    ('logreg', LogisticRegression(max_iter=1000))
])

model.fit(X_train_3, Y_train)
predictions = model.predict(X_test_3)
get_results(Y_test, predictions)

c:\Main_Doc\M1\S2\TAL\projet\tal_project\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['neuf', 'qu', 'quelqu'] not in stop_words.
  warnings.warn(


Resultat (Micro-F1) : 0.88
Résultat (Macro-F1) : 0.87
Résultat (Accuracy) : 0.88
Classes : ['Dessert', 'Entrée', 'Plat principal']
Résultat (F1 par classe) : [0.98663426 0.7327044  0.87623386]
Matrice de confusion : [[406   1   0]
 [  8 233  96]
 [  2  65 577]]
